# Day 7 — Segmentation, CLV, Emotion

**Phase:** plan's Day 9–10 (segmentation + CLV), adjusted for the retention reality, plus the Day 6 emotion deferral.

### The reframe, stated up front (so it survives an interview)
Day 6 measured **3.1% repeat purchase**. That breaks the textbook plan:
- **RFM's Frequency is degenerate** (one value for 97% of customers). We compute it, show the collapse, then drop it.
- **Predictive CLV (BG/NBD) cannot fit** on a 97%-single-purchase base. We do **descriptive** CLV (observed value per segment) instead.
- Segmentation runs on features that actually vary: **monetary, recency, delivery experience, lateness.**

### Order of work
1. Build customer-level feature table (from master, keyed on `customer_unique_id`).
2. RFM, and the F-collapse made explicit.
3. K-means segmentation on features with real variance.
4. Descriptive CLV per segment.
5. Emotion on negative reviews (HF model, keyword fallback when HF is offline).

Outputs feed the dashboard segment view (Day 11) and the Day 8 repeat-purchase model.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

# Auto-detect project root whether running from notebooks/ or project root
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
assert (ROOT / 'data' / 'processed').exists(), f"data/processed not found from ROOT={ROOT}"
PROC = ROOT / 'data' / 'processed'

master = pd.read_parquet(PROC / 'olist_master.parquet')
scored = pd.read_parquet(PROC / 'olist_reviews_scored.parquet')
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])

print('master:', master.shape, '| scored:', scored.shape)

master: (99441, 31) | scored: (11997, 8)


## 1. Customer-level feature table

One row per real person (`customer_unique_id`), not per order. Recency is measured from the day after the
last purchase in the dataset, the standard RFM convention.

In [2]:
ref_date = master['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print('Reference date for recency:', ref_date.date())

cust = (master.groupby('customer_unique_id')
        .agg(frequency=('order_id', 'nunique'),
             monetary=('total_payment', 'sum'),
             last_purchase=('order_purchase_timestamp', 'max'),
             avg_delivery_days=('delivery_time_days', 'mean'),
             late_rate=('is_late', 'mean'),
             avg_review=('review_score', 'mean'))
        .reset_index())

cust['recency_days'] = (ref_date - cust['last_purchase']).dt.days
cust['monetary'] = cust['monetary'].fillna(0)

print('customers:', len(cust))
cust.describe()

Reference date for recency: 2018-10-18
customers: 96096


,frequency,monetary,last_purchase,avg_delivery_days,late_rate,avg_review,recency_days
count,96096.000000,96096.000000,96096,93356.000000,96096.00000,95380.000000,96096.000000
mean,1.034809,166.592492,2018-01-02 12:40:19.655864832,12.103324,0.06610,4.084845,288.735691
min,1.000000,0.000000,2016-09-04 21:15:19,0.000000,0.00000,1.000000,1.000000
25%,1.000000,63.120000,2017-09-15 09:04:17.249999872,6.000000,0.00000,4.000000,164.000000
50%,1.000000,108.000000,2018-01-21 19:39:16,10.000000,0.00000,5.000000,269.000000
75%,1.000000,183.530000,2018-05-06 20:14:49.750000128,15.000000,0.00000,5.000000,398.000000
max,17.000000,13664.080000,2018-10-17 17:30:18,209.000000,1.00000,5.000000,773.000000
std,0.214384,231.428332,NaN,9.551802,0.24699,1.341971,153.414676


## 2. RFM, and the Frequency collapse

R and M score cleanly into quintiles. F does not, because >97% of customers have exactly one order.
The `try/except` below proves it instead of asserting it. **This is a finding, not a bug.**

In [3]:
# R: lower recency is better, so reverse the labels (5 = most recent)
cust['R_score'] = pd.qcut(cust['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
# M: higher spend is better
cust['M_score'] = pd.qcut(cust['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

print('Frequency value counts (the whole problem in one table):')
print(cust['frequency'].value_counts().head())
print()
try:
    cust['F_score'] = pd.qcut(cust['frequency'], 5, labels=[1, 2, 3, 4, 5])
    print('F qcut succeeded (unexpected for Olist).')
except ValueError as e:
    print('F qcut FAILED, as expected:', str(e).split(chr(10))[0])
    print('Verdict: Frequency has no usable spread. Dropping F from segmentation.')

Frequency value counts (the whole problem in one table):
frequency
1    93099
2     2745
3      203
4       30
5        8
Name: count, dtype: int64

F qcut FAILED, as expected: Bin edges must be unique: Index([1.0, 1.0, 1.0, 1.0, 1.0, 17.0], dtype='float64', name='frequency').
Verdict: Frequency has no usable spread. Dropping F from segmentation.


### RM grid (the RFM we can actually defend)
With F dead, the honest summary is a Recency x Monetary grid. Repeat customers are a tiny named cohort handled separately.

In [4]:
rm = pd.crosstab(cust['R_score'], cust['M_score'])
print('Customer counts by Recency (rows) x Monetary (cols) score:')
print(rm)
print()
print(f"Repeat cohort (2+ orders): {(cust['frequency'] > 1).sum():,} customers "
      f"({(cust['frequency'] > 1).mean()*100:.2f}%). Tracked separately, too small to segment on.")

Customer counts by Recency (rows) x Monetary (cols) score:
M_score     1     2     3     4     5
R_score                              
1        3984  4041  3802  3485  3818
2        3697  4147  3824  3724  3908
3        3951  3716  3789  3945  3642
4        3748  3651  3921  4066  3913
5        3840  3664  3883  3999  3938

Repeat cohort (2+ orders): 2,997 customers (3.12%). Tracked separately, too small to segment on.


## 3. K-means segmentation

Features chosen for real variance and business meaning:
- `log_monetary` (spend, log to tame the right skew)
- `recency_days`
- `avg_delivery_days` (delivery experience; median-imputed for never-delivered)
- `late_rate`

Frequency is excluded on purpose (no variance). Standardize, then pick k with elbow + silhouette.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

seg = cust.copy()
seg['avg_delivery_days'] = seg['avg_delivery_days'].fillna(seg['avg_delivery_days'].median())
seg['late_rate'] = seg['late_rate'].fillna(0)
seg['log_monetary'] = np.log1p(seg['monetary'])

feats = ['log_monetary', 'recency_days', 'avg_delivery_days', 'late_rate']
X = StandardScaler().fit_transform(seg[feats])
print('feature matrix:', X.shape)

feature matrix: (96096, 4)


In [6]:
# Elbow (inertia) + silhouette. Silhouette is sampled, full pairwise on 96k would blow memory.
inertia, sil = [], []
ks = range(2, 8)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_, sample_size=10000, random_state=42))

for k, i, s in zip(ks, inertia, sil):
    print(f"k={k}  inertia={i:>12,.0f}  silhouette={s:.3f}")

k=2  inertia=     254,032  silhouette=0.607
k=3  inertia=     188,412  silhouette=0.301
k=4  inertia=     147,161  silhouette=0.299
k=5  inertia=     131,266  silhouette=0.269
k=6  inertia=     116,468  silhouette=0.287
k=7  inertia=     106,377  silhouette=0.286


Pick k from the output above: look for the elbow in inertia and the peak silhouette. For Olist this is
usually **k=4**. Change `K` below if your numbers point elsewhere, don't just trust the default.

In [7]:
K = 4   # <-- set from elbow + silhouette above
km = KMeans(n_clusters=K, random_state=42, n_init=10).fit(X)
seg['cluster'] = km.labels_

# Profile clusters on RAW (un-scaled) features so the numbers are readable
profile = (seg.groupby('cluster')
           .agg(customers=('customer_unique_id', 'size'),
                avg_monetary=('monetary', 'mean'),
                avg_recency=('recency_days', 'mean'),
                avg_delivery_days=('avg_delivery_days', 'mean'),
                late_rate=('late_rate', 'mean'),
                avg_review=('avg_review', 'mean'))
           .round(2))
profile['pct_of_base'] = (profile['customers'] / len(seg) * 100).round(1)
profile

,customers,avg_monetary,avg_recency,avg_delivery_days,late_rate,avg_review,pct_of_base
cluster,,,,,,,
0,34611,72.56,188.18,8.94,0.00,4.30,36.0
1,27065,335.15,230.79,12.27,0.00,4.12,28.2
2,6323,179.24,270.35,33.53,0.99,2.27,6.6
3,28097,117.22,472.56,10.82,0.00,4.19,29.2


### Name the segments

Read the profile above and name each cluster by what it actually is. The template below is a *guess* based
on typical Olist clustering. **Verify every name against the real profile numbers before trusting it.**
A name that doesn't match the row is worse than no name.

In [8]:
# EDIT after reading the profile. Keys are cluster ids (0..K-1).
segment_names = {
    0: 'High-value, satisfied',
    1: 'Low-value, on-time',
    2: 'Late delivery, unhappy',
    3: 'Older / dormant',
}
seg['segment'] = seg['cluster'].map(segment_names)
print(seg['segment'].value_counts())

segment
High-value, satisfied     34611
Older / dormant           28097
Low-value, on-time        27065
Late delivery, unhappy     6323
Name: count, dtype: int64


## 4. Descriptive CLV per segment

Observed historical value, not a predicted lifetime. Since ~97% buy once, CLV here is effectively
realized value per customer. Predictive CLV (BG/NBD, Gamma-Gamma) is deliberately not attempted:
it needs a repeat-purchase base this dataset doesn't have. State that plainly in the report.

In [9]:
clv = (seg.groupby('segment')
       .agg(customers=('customer_unique_id', 'size'),
            avg_observed_value=('monetary', 'mean'),
            total_observed_value=('monetary', 'sum'),
            avg_orders=('frequency', 'mean'),
            repeat_rate_pct=('frequency', lambda f: (f > 1).mean() * 100),
            avg_review=('avg_review', 'mean'),
            late_rate=('late_rate', 'mean'))
       .round(2)
       .sort_values('total_observed_value', ascending=False))
clv

,customers,avg_observed_value,total_observed_value,avg_orders,repeat_rate_pct,avg_review,late_rate
segment,,,,,,,
"Low-value, on-time",27065,335.15,9070919.69,1.08,6.98,4.12,0.00
Older / dormant,28097,117.22,3293403.46,1.02,2.11,4.19,0.00
"High-value, satisfied",34611,72.56,2511204.20,1.01,1.10,4.30,0.00
"Late delivery, unhappy",6323,179.24,1133344.77,1.02,2.12,2.27,0.99


## 5. Emotion on negative reviews

Finishes the VoC layer. Tries the HF emotion model first
(`j-hartmann/emotion-english-distilroberta-base`). **It will likely fail the same way BERTopic did**
(HF offline / SSL), so a keyword tagger is the fallback and is what most likely ends up in the dashboard.
Label it honestly as rule-based if the transformer doesn't run.

In [12]:
j = scored.merge(master[['order_id', 'is_late']], on='order_id', how='inner')
neg = j[(j['sentiment_pred'] == 'negative') | (j['review_score'] <= 2)].copy()
neg_text = neg['review_clean'].fillna('').astype(str)
print(f"Negative reviews for emotion tagging: {len(neg):,}")

Negative reviews for emotion tagging: 10,692


In [13]:
emotion_source = None
try:
    from transformers import pipeline
    clf = pipeline('text-classification',
                   model='j-hartmann/emotion-english-distilroberta-base',
                   top_k=1, truncation=True)
    out = clf(neg_text.tolist(), batch_size=32)
    neg['emotion'] = [o[0]['label'] for o in out]
    emotion_source = 'transformer (j-hartmann/emotion-english-distilroberta-base)'
except Exception as e:
    print('HF emotion model unavailable, using keyword fallback. Reason:')
    print(str(e).split(chr(10))[0])


HF emotion model unavailable, using keyword fallback. Reason:
We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like j-hartmann/emotion-english-distilroberta-base is not the path to a directory containing a file named config.json.


C:\Users\akskumari\Desktop\cx-analytics-project\venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
# Keyword fallback (runs only if the transformer didn't tag)
if 'emotion' not in neg.columns:
    lex = {
        'anger':          ['terrible', 'awful', 'worst', 'horrible', 'absurd', 'unacceptable',
                           'scam', 'cheated', 'ridiculous', 'never buy', 'lie', 'liar'],
        'frustration':    ['still waiting', "haven't", 'hasn', 'never received', "didn't receive",
                           'no response', 'no answer', 'delayed', 'late', 'no contact', 'where is'],
        'disappointment': ['poor quality', 'disappointed', 'expected', 'not as described',
                           'regret', 'let down', 'bad quality', 'broke', 'broken', 'defective'],
    }
    def tag(t):
        t = t.lower()
        for emo, kws in lex.items():
            if any(k in t for k in kws):
                return emo
        return 'frustration'   # default: delivery complaints dominate
    neg['emotion'] = neg_text.apply(tag)
    emotion_source = 'keyword lexicon (rule-based)'

print('Emotion source:', emotion_source)
print()
print(neg['emotion'].value_counts())

Emotion source: keyword lexicon (rule-based)

emotion
frustration       9349
disappointment     673
anger              670
Name: count, dtype: int64


## 6. Save outputs

In [15]:
seg_out = seg[['customer_unique_id', 'frequency', 'monetary', 'recency_days',
               'avg_delivery_days', 'late_rate', 'avg_review',
               'R_score', 'M_score', 'cluster', 'segment']]
seg_out.to_parquet(PROC / 'customer_segments.parquet', index=False)

clv.to_csv(PROC / 'segment_clv.csv')
neg[['order_id', 'review_clean', 'sentiment_pred', 'theme_pred', 'review_score', 'emotion']] \
    .to_parquet(PROC / 'reviews_with_emotion.parquet', index=False)

print('Saved to', PROC)
print(' - customer_segments.parquet   (', len(seg_out), 'customers )')
print(' - segment_clv.csv')
print(' - reviews_with_emotion.parquet (', len(neg), 'negative reviews,', emotion_source, ')')

Saved to C:\Users\akskumari\Desktop\cx-analytics-project\data\processed
 - customer_segments.parquet   ( 96096 customers )
 - segment_clv.csv
 - reviews_with_emotion.parquet ( 10692 negative reviews, keyword lexicon (rule-based) )


## Done — hand to Day 8

- `customer_segments.parquet` is a feature input for the **Day 8 repeat-purchase model** and the dashboard segment view.
- `reviews_with_emotion.parquet` feeds the dashboard VoC emotion breakdown.
- `segment_clv.csv` is the segment value table for the report.

**Day 8:** repeat-purchase prediction (binary "did they ever order again"), LR then XGBoost, on the honest
label. Features: delivery time, late flag, monetary, review score, segment. Report ROC-AUC and feature
importance. This is the resume's "churn model" reframed onto a target that exists in the data.
